# 04 · BTS flights → UTC event times → hourly demand and throughput
For each study airport:
1. Pull the flights that depart from or arrive at it out of the monthly BTS files.
2. Convert every event to UTC. BTS times are *local* to the airport where the event happens.
   - **Scheduled departure:** `FL_DATE + CRS_DEP_TIME` in the origin's timezone.
   - **Scheduled arrival:** scheduled departure + `CRS_ELAPSED_TIME`. This avoids midnight and timezone rollover bugs.
   - **Wheels-off:** `FL_DATE + WHEELS_OFF` in the origin's timezone. Add 1 day if that lands more than 6 h before the scheduled departure (delays past midnight).
   - **Wheels-on:** wheels-off + `AIR_TIME`.
3. Aggregate to airport-hours. Demand is counted by *scheduled* hour, and throughput by *wheels-off/on* hour, i.e. runway time.

Outputs: `data/interim/flights_{APT}.parquet` (per flight) and `hourly_flights_{APT}.parquet`.

In [1]:
%run ../config.ipynb

ROOT = /Users/dylannegvesky/Documents/FSU Documents/Fall 2026/ProjDS/PDS_Final
study airports: ['EWR'] | BTS 2023-01 → latest | train ≤ 2025-12


In [2]:
airports = pd.read_parquet(INTERIM / "airports.parquet")
TZ = dict(zip(airports["iata_id"], airports["tz"])) | dict(zip(airports["faa_id"], airports["tz"]))
STUDY = list(STUDY_AIRPORTS)
bts_files = sorted(BTS_DIR.glob("*.parquet"))
print(f"{len(bts_files)} BTS months: {bts_files[0].stem} → {bts_files[-1].stem}")

COLS = ["FL_DATE", "OP_UNIQUE_CARRIER", "TAIL_NUM", "OP_CARRIER_FL_NUM", "ORIGIN", "DEST",
        "CRS_DEP_TIME", "WHEELS_OFF", "WHEELS_ON", "CRS_ELAPSED_TIME", "AIR_TIME",
        "TAXI_OUT", "TAXI_IN", "DEP_DELAY", "ARR_DELAY", "DEP_DEL15", "ARR_DEL15",
        "CANCELLED", "CANCELLATION_CODE", "DIVERTED",
        "CARRIER_DELAY", "WEATHER_DELAY", "NAS_DELAY", "SECURITY_DELAY", "LATE_AIRCRAFT_DELAY"]

flights = pd.concat(
    [pd.read_parquet(p, columns=COLS, filters=[[("ORIGIN", "in", STUDY)], [("DEST", "in", STUDY)]])
     for p in bts_files], ignore_index=True)
print(f"{len(flights):,} flights touching {STUDY}")

45 BTS months: 2023_01 → 2026_09


956,449 flights touching ['EWR']


## Local → UTC

In [3]:
def hhmm_minutes(s):
    s = s.astype("string").str.zfill(4)
    return pd.to_numeric(s.str[:2], errors="coerce") * 60 + pd.to_numeric(s.str[2:], errors="coerce")  # 2400 → 1440 = next day 00:00


def local_to_utc(date, minutes, tz_series):
    """date (naive midnight) + minutes, interpreted in each row's IANA tz → UTC."""
    naive = date + pd.to_timedelta(minutes, unit="min")
    out = pd.Series(pd.NaT, index=naive.index, dtype="datetime64[ns, UTC]")
    for tz, idx in tz_series.groupby(tz_series).groups.items():
        local = naive.loc[idx].dt.tz_localize(tz, ambiguous=np.zeros(len(idx), dtype=bool),
                                              nonexistent="shift_forward")
        out.loc[idx] = local.dt.tz_convert("UTC")
    return out


f = flights
f["origin_tz"] = f["ORIGIN"].map(TZ)
f["dest_tz"] = f["DEST"].map(TZ)
assert f["origin_tz"].notna().all() and f["dest_tz"].notna().all(), "unknown airport timezone, so rerun 02_nasr"

date = pd.to_datetime(f["FL_DATE"]).dt.normalize()
f["sched_dep_utc"] = local_to_utc(date, hhmm_minutes(f["CRS_DEP_TIME"]), f["origin_tz"])
f["sched_arr_utc"] = f["sched_dep_utc"] + pd.to_timedelta(f["CRS_ELAPSED_TIME"], unit="min")

off = local_to_utc(date, hhmm_minutes(f["WHEELS_OFF"]), f["origin_tz"])
off = off.where(off >= f["sched_dep_utc"] - pd.Timedelta(hours=6), off + pd.Timedelta(days=1))
f["wheels_off_utc"] = off
f["wheels_on_utc"] = off + pd.to_timedelta(f["AIR_TIME"], unit="min")
f["nas_wx_delay"] = f["NAS_DELAY"].fillna(0) + f["WEATHER_DELAY"].fillna(0)
f["cancel_wx_nas"] = (f["CANCELLED"] == 1) & f["CANCELLATION_CODE"].isin(["B", "C"])

print("missing scheduled times:", f["sched_dep_utc"].isna().sum(), f["sched_arr_utc"].isna().sum())
operated = (f["CANCELLED"] == 0)
print("operated flights missing wheels-off:", f.loc[operated, "wheels_off_utc"].isna().sum())
print("non-diverted operated flights missing wheels-on:",
      f.loc[operated & (f["DIVERTED"] == 0), "wheels_on_utc"].isna().sum())

missing scheduled times: 0 0
operated flights missing wheels-off: 0
non-diverted operated flights missing wheels-on: 0


Sanity check: the computed wheels-on should match BTS's own local `WHEELS_ON` clock time at the destination, to within a minute or two of rounding.

In [4]:
chk = f.dropna(subset=["wheels_on_utc", "WHEELS_ON"]).sample(min(20000, len(f)), random_state=0)
local_on = pd.Series([t.tz_convert(z) for t, z in zip(chk["wheels_on_utc"], chk["dest_tz"])], index=chk.index)
calc_hhmm = local_on.map(lambda t: t.hour * 60 + t.minute)
bts_hhmm = hhmm_minutes(chk["WHEELS_ON"]) % 1440
err = ((calc_hhmm - bts_hhmm + 720) % 1440 - 720).abs()
print(f"wheels-on reconstruction: {(err <= 2).mean():.2%} within 2 min, {(err > 15).mean():.2%} off by >15 min")

wheels-on reconstruction: 100.00% within 2 min, 0.00% off by >15 min


## Hourly aggregation per airport
- `sched_dep` / `sched_arr`: **demand**, every scheduled flight including later cancellations.
- `expected_*_nc`: flights expected at the runway in the hour, excluding cancellations (and diversions, for arrivals). Backlog uses these, because a cancelled flight never uses the runway.
- `expected_*_all`: the same expected runway hour for **every** scheduled flight, cancellations included. This is what the schedule alone says, so it's known in advance. 06 uses it for next-hour demand.
- `actual_dep` / `actual_arr`: **throughput**, by wheels-off/on hour.
- `arr_nas_wx_delayed`: arrivals scheduled in the hour with NAS + weather delay ≥ 15 min. BTS records delay causes on the arrival, so for this airport's *arrivals* they reflect conditions here.
- `taxi_out_mean`: by wheels-off hour. Departure queues show up here, since a departure's NAS delay may come from the destination instead.
- **Backlog** at the end of hour t: flights that *should* have used the runway by now minus flights that did, cumulative since the start of the operating day (04:00 local), plus a rolling 4-hour version that doesn't reset.
  - "Should have" means the **expected runway time**: scheduled departure + the airport's median taxi-out, and scheduled arrival − its median taxi-in. Both medians come from training months only.
  - Without this shift, the backlog would build up over every normal day just from the gap between gate times and runway times.

In [5]:
def hourly_for(apt):
    tz = TZ[apt]
    dep = f[f["ORIGIN"] == apt]
    arr = f[f["DEST"] == apt]
    fl = lambda s: s.dt.floor("h")
    dep_c, arr_c = dep["CANCELLED"] == 1, arr["CANCELLED"] == 1
    arr_div = arr["DIVERTED"] == 1
    train_cut = pd.Period(TRAIN_END, "M").end_time.tz_localize("UTC")
    taxi_out_med = dep.loc[dep["sched_dep_utc"] <= train_cut, "TAXI_OUT"].median()
    taxi_in_med = arr.loc[arr["sched_arr_utc"] <= train_cut, "TAXI_IN"].median()
    print(f"{apt}: median taxi-out {taxi_out_med:.0f} min, taxi-in {taxi_in_med:.0f} min (training months)")
    exp_off = dep["sched_dep_utc"] + pd.Timedelta(minutes=taxi_out_med)
    exp_on = arr["sched_arr_utc"] - pd.Timedelta(minutes=taxi_in_med)

    parts = {
        "sched_dep": dep.groupby(fl(dep["sched_dep_utc"])).size(),
        "sched_arr": arr.groupby(fl(arr["sched_arr_utc"])).size(),
        "expected_dep_all": fl(exp_off).value_counts(),
        "expected_arr_all": fl(exp_on).value_counts(),
        "expected_dep_nc": fl(exp_off[~dep_c]).value_counts(),
        "expected_arr_nc": fl(exp_on[~arr_c & ~arr_div]).value_counts(),
        "actual_dep": fl(dep.loc[~dep_c, "wheels_off_utc"]).value_counts(),
        "actual_arr": fl(arr.loc[~arr_c & ~arr_div, "wheels_on_utc"]).value_counts(),
        "cancel_dep": dep[dep_c].groupby(fl(dep.loc[dep_c, "sched_dep_utc"])).size(),
        "cancel_arr": arr[arr_c].groupby(fl(arr.loc[arr_c, "sched_arr_utc"])).size(),
        "cancel_wx_nas": pd.concat([dep.loc[dep["cancel_wx_nas"], "sched_dep_utc"],
                                    arr.loc[arr["cancel_wx_nas"], "sched_arr_utc"]]).dt.floor("h").value_counts(),
        "diverted_arr": arr[arr_div].groupby(fl(arr.loc[arr_div, "sched_arr_utc"])).size(),
        "arr_nas_wx_delayed": arr[arr["nas_wx_delay"] >= NAS_WX_DELAY_MIN]
                                 .groupby(fl(arr.loc[arr["nas_wx_delay"] >= NAS_WX_DELAY_MIN, "sched_arr_utc"])).size(),
        "arr_del15": arr.groupby(fl(arr["sched_arr_utc"]))["ARR_DEL15"].sum(),
        "dep_del15": dep.groupby(fl(dep["sched_dep_utc"]))["DEP_DEL15"].sum(),
        "arr_delay_mean": arr.groupby(fl(arr["sched_arr_utc"]))["ARR_DELAY"].mean(),
        "dep_delay_mean": dep.groupby(fl(dep["sched_dep_utc"]))["DEP_DELAY"].mean(),
        "taxi_out_mean": dep.groupby(fl(dep["wheels_off_utc"]))["TAXI_OUT"].mean(),
        "taxi_out_median": dep.groupby(fl(dep["wheels_off_utc"]))["TAXI_OUT"].median(),
        "taxi_in_mean": arr.groupby(fl(arr["wheels_on_utc"]))["TAXI_IN"].mean(),
    }
    h = pd.DataFrame(parts)
    start = pd.Period(BTS_START, "M").start_time.tz_localize(tz).tz_convert("UTC")
    end = (pd.to_datetime(f["FL_DATE"]).max().normalize() + pd.Timedelta(days=1)).tz_localize(tz).tz_convert("UTC")
    spine = pd.date_range(start, end, freq="h", tz="UTC", name="hour_utc", inclusive="left")
    h = h.reindex(spine)
    count_cols = [c for c in h if not c.endswith(("_mean", "_median"))]
    h[count_cols] = h[count_cols].fillna(0).astype(int)

    h["sched_total"] = h["sched_dep"] + h["sched_arr"]
    h["actual_total"] = h["actual_dep"] + h["actual_arr"]
    h["expected_total_all"] = h["expected_dep_all"] + h["expected_arr_all"]
    net = (h["expected_dep_nc"] + h["expected_arr_nc"]) - h["actual_total"]
    local = h.index.tz_convert(tz)
    ops_day = (local - pd.Timedelta(hours=OPS_DAY_START_LOCAL_HOUR)).date
    h["backlog"] = net.groupby(ops_day).cumsum()
    h["backlog_dep"] = (h["expected_dep_nc"] - h["actual_dep"]).groupby(ops_day).cumsum()
    h["backlog_arr"] = (h["expected_arr_nc"] - h["actual_arr"]).groupby(ops_day).cumsum()
    h["backlog_roll4"] = net.rolling(4, min_periods=1).sum()
    h["backlog_growth"] = h["backlog"].diff().fillna(h["backlog"])  # first hour of the day: growth = level

    # Excess taxi-out vs. this airport's typical value for the same local hour and month,
    # with the baseline computed from TRAIN months only to avoid peeking at test data.
    key = pd.DataFrame({"hr": local.hour, "mo": local.month}, index=h.index)
    train = h.index < pd.Period(TRAIN_END, "M").end_time.tz_localize(tz)
    base = h.loc[train, "taxi_out_mean"].groupby([key.loc[train, "hr"], key.loc[train, "mo"]]).median()
    h["taxi_out_baseline"] = pd.MultiIndex.from_frame(key).map(base.to_dict().get).astype(float)
    h["taxi_out_excess"] = h["taxi_out_mean"] - h["taxi_out_baseline"]

    h["local_time"] = local.tz_localize(None)
    h.insert(0, "airport", apt)
    return h.reset_index()


hourly = {}
for apt in STUDY:
    hourly[apt] = hourly_for(apt)
    keep = (f["ORIGIN"] == apt) | (f["DEST"] == apt)
    f.loc[keep].to_parquet(INTERIM / f"flights_{apt}.parquet", index=False)
    hourly[apt].to_parquet(INTERIM / f"hourly_flights_{apt}.parquet", index=False)
    print(f"{apt}: {keep.sum():,} flights → {len(hourly[apt]):,} hours")

EWR: median taxi-out 21 min, taxi-in 9 min (training months)


EWR: 956,449 flights → 32,855 hours


## Checks

In [6]:
for apt, h in hourly.items():
    print(f"== {apt}")
    # 1. Daily totals: actual throughput equals operated flights
    dep = f[(f["ORIGIN"] == apt) & (f["CANCELLED"] == 0)]
    arr = f[(f["DEST"] == apt) & (f["CANCELLED"] == 0) & (f["DIVERTED"] == 0)]
    print("actual_dep total vs operated departures:", h["actual_dep"].sum(), len(dep))
    print("actual_arr total vs landed arrivals:   ", h["actual_arr"].sum(), len(arr))
    # 2. Backlog returns to about 0 overnight (04:00 local)
    overnight = h[h["local_time"].dt.hour == OPS_DAY_START_LOCAL_HOUR - 1]["backlog"]
    print("backlog at 03:00 local, quantiles:", overnight.quantile([0.05, 0.5, 0.95]).to_dict())
    # 3. DST changeovers: the UTC spine has no gaps or duplicates
    assert h["hour_utc"].is_unique and h["hour_utc"].diff().dropna().eq(pd.Timedelta("1h")).all()
    # 4. Typical day profile (local hour)
    prof = h.groupby(h["local_time"].dt.hour)[["sched_dep", "sched_arr", "actual_dep", "actual_arr", "backlog"]].mean()
    display(prof.round(1).T)

== EWR
actual_dep total vs operated departures: 464476 464476
actual_arr total vs landed arrivals:    462259 462289
backlog at 03:00 local, quantiles: {0.05: -3.0, 0.5: 0.0, 0.95: 3.0}


local_time,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23
sched_dep,0.0,0.0,0.0,0.0,0.0,5.8,27.9,27.7,20.0,22.9,21.4,18.8,13.1,18.3,20.9,21.7,23.2,20.6,22.7,22.3,20.1,17.9,3.8,0.1
sched_arr,6.5,1.7,0.1,0.0,1.2,9.3,6.1,20.3,19.2,17.9,18.0,17.8,12.7,17.9,21.6,22.1,21.3,22.8,18.5,19.5,22.4,19.8,13.7,19.1
actual_dep,1.7,0.7,0.3,0.1,0.0,4.0,22.7,23.4,21.0,20.7,21.3,19.0,17.1,16.2,17.1,19.4,19.6,19.9,18.1,20.9,20.1,18.1,13.6,4.2
actual_arr,9.4,3.4,1.3,1.0,3.6,7.2,9.8,18.5,19.5,18.9,16.0,16.2,14.9,18.7,19.4,19.2,18.2,20.0,18.1,17.6,19.1,17.2,15.3,15.3
backlog,5.9,2.5,1.1,-0.0,-1.1,0.2,-2.4,0.3,3.4,2.4,3.9,3.1,2.4,3.5,4.6,8.3,9.1,13.9,14.5,16.8,20.3,17.4,13.8,10.0


Manual count for one day: hourly scheduled departures in local time, straight from the raw rows, compared with the pipeline.

In [7]:
apt = STUDY[0]
day = pd.to_datetime(f["FL_DATE"]).min().normalize() + pd.Timedelta(days=14)
raw = f[(f["ORIGIN"] == apt) & (pd.to_datetime(f["FL_DATE"]) == day)]
manual = (hhmm_minutes(raw["CRS_DEP_TIME"]) // 60).value_counts().sort_index()
h = hourly[apt]
d = h[h["local_time"].dt.normalize() == day]
pipe = d.set_index(d["local_time"].dt.hour)["sched_dep"]
print("check day:", day.date())
cmp = pd.DataFrame({"manual_local": manual, "pipeline": pipe}).fillna(0).astype(int)
display(cmp.T)
assert (cmp["manual_local"] == cmp["pipeline"]).all()

check day: 2023-01-15


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23
manual_local,0,0,0,0,0,7,24,26,25,22,21,22,17,16,20,25,22,27,30,23,23,19,5,0
pipeline,0,0,0,0,0,7,24,26,25,22,21,22,17,16,20,25,22,27,30,23,23,19,5,0
